# Model for job Recommendaion

## 1. Import Libraries

In [1]:
import kagglehub
import pandas as pd
import ast
import random

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import MiniBatchKMeans

c:\Users\computer\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


##  2. Data Loading

In [2]:
path = kagglehub.dataset_download("hayaalwizrah1/job-recommendation")

data = pd.read_csv(path + "/linkedin-jobs-and-skills.csv")

df = data.copy()

In [3]:
print(df.shape)
df.head()

(1294132, 12)


,job_title,company,job_location,first_seen,search_city,search_country,search_position,job_level,job_type,job_skills,job_title_c,job_skills_c
0,Account Executive - Dispensing (Norcal/Norther...,BD,"San Diego, CA",2024-01-15,Coronado,United States,Color Maker,Mid Senior,Onsite,"Medical equipment sales, Key competitors, Term...",Account Executive - Dispensing - Becton Dickinson,"['medical sales', 'key competitors', 'terminol..."
1,Registered Nurse - Rn Care Manager,Trinity Health MI,"Norton Shores, MI",2024-01-14,Grand Haven,United States,Director Nursing Service,Mid Senior,Onsite,"Nursing, Bachelor of Science in Nursing, Maste...",Nurse Care Manager,"['nursing', ""master's degree in nursing"", 'pat..."
2,Restaurant Supervisor - The Forklift,Wasatch Adaptive Sports,"Sandy, UT",2024-01-14,Tooele,United States,Stand-In,Mid Senior,Onsite,"Restaurant Operations Management, Inventory Ma...",Restaurant Supervisor,"['restaurant operations', 'inventory managemen..."
3,Independent Real Estate Agent,Howard Hanna | Rand Realty,"Englewood Cliffs, NJ",2024-01-16,Pinehurst,United States,Real-Estate Clerk,Mid Senior,Onsite,"Real Estate, Customer Service, Sales, Negotiat...",Independent Real Estate Agent,"['real estate', 'customer service', 'sales', '..."
4,Registered Nurse (Rn),Trinity Health MI,"Muskegon, MI",2024-01-14,Muskegon,United States,Nurse Practitioner,Mid Senior,Onsite,"Nursing, BSN, Medical License, Virtual RN, Nur...",Registered Nurse,"['nursing', 'bsn', 'medical license', 'virtual..."


In [4]:
df = df.drop(columns=['job_title', 'company' ,'job_location', 'first_seen', 'search_city', 'search_country', 'search_position',	'job_level', 'job_type', 'job_skills'])

In [5]:
df.columns

Index(['job_title_c', 'job_skills_c'], dtype='str')

In [6]:
df = df.rename(columns={"job_title_c": "job_title", "job_skills_c": "job_skills"})

In [7]:
df['job_skills'] = df['job_skills'].apply(ast.literal_eval)

print(type(df['job_skills'].iloc[0]))

<class 'list'>


##  3. TF-IDF

In [ ]:
# 1: Remove junk single-character skills 
junk_single_chars = {'h', 's', '3', 'e', 'n', 'j', 'a', 'm', '5', '9', 'z', 'p', 'x', 'k',
                      'i', '*', '2', 'f', '+', '$', '6', 'g', '>', 'q', 't', 'b', 'd', 'o'}

def clean_skills(skills_list):
    return [s for s in skills_list if s not in junk_single_chars]

df['job_skills'] = df['job_skills'].apply(clean_skills)

# 2: Convert skills list to text 
def format_skill(skill):
    if skill == "r":
        return "lang_r"
    if skill == "c":
        return "lang_c"
    return skill.replace(" ", "_")

df["skills_text"] = df["job_skills"].apply(
    lambda skills: " ".join([format_skill(s) for s in skills])
)

# 3: Build TF-IDF 

tfidf = TfidfVectorizer(min_df=15)
tfidf_matrix = tfidf.fit_transform(df["skills_text"])

print("Final shape:", tfidf_matrix.shape)

Final shape: (1294132, 64088)


In [9]:
def prepare_query(user_skills):
    """
    Converts a list of user skills into a TF-IDF-ready vector.
    Used by all recommendation models to ensure consistent preprocessing.
    """
    query = " ".join([format_skill(skill.strip().lower()) for skill in user_skills])
    return tfidf.transform([query])

In [10]:
test_cases = {
    "python, sql, excel": "Data Analyst",
    "machine learning, deep learning, nlp": "Machine Learning Engineer",
    "react, javascript, css": "Frontend Developer",
    "nursing, patient care": "Registered Nurse",
    "accounting, excel, financial analysis": "Accountant",
    "marketing, social media, content": "Marketing Specialist",
    "java, spring boot, sql": "Backend Developer",
    "sales, negotiation, crm": "Sales Representative",
    "graphic design, photoshop, illustrator": "Graphic Designer",
    "project management, agile, scrum": "Project Manager",
}

In [11]:
random.seed(42)

sample_indices = random.sample(range(len(df)), 30)

extra_test_cases = {}
for idx in sample_indices:
    job_title = df.iloc[idx]['job_title']
    skills = df.iloc[idx]['job_skills']
    sample_skills = random.sample(skills, min(3, len(skills)))
    skills_text = ", ".join(sample_skills)
    extra_test_cases[skills_text] = job_title

test_cases.update(extra_test_cases)
print(f"Total test cases now: {len(test_cases)}")

Total test cases now: 40


## 4. Modeling

Three recommendation approaches were implemented and compared using the same TF-IDF representation.

### 4.1 Nearest Neighbors

In [12]:
nn = NearestNeighbors(n_neighbors=10, metric='cosine')
nn.fit(tfidf_matrix)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",10
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [13]:
# NearestNeighbors(test)
def calculate_precision_nn(nn_model, test_cases, top_n=5):
    correct = 0
    for skills_text, expected_title in test_cases.items():
        user_skills = [s.strip() for s in skills_text.split(",")]
        query_vec = prepare_query(user_skills)                                
        distances, indices = nn_model.kneighbors(query_vec, n_neighbors=top_n)
        top_titles = [df.iloc[idx]["job_title"] for idx in indices[0]]
        if any(expected_title.lower() in title.lower() for title in top_titles):
            correct += 1
    return correct / len(test_cases) * 100


In [14]:
# final model NearestNeighbors
def recommend_jobs_nn(user_skills, top_n=10):
    """
    Takes a list of user skills and returns the top matching job titles.
    Parameters:
        user_skills (list of str): e.g. ["python", "machine learning", "nlp"]
        top_n (int): number of recommendations to return
    Returns:
        pandas.DataFrame with columns: job_title, similarity
    """
    query_vec = prepare_query(user_skills)
    distances, indices = nn.kneighbors(query_vec, n_neighbors=top_n)
    results = []
    for idx, dist in zip(indices[0], distances[0]):
        results.append({
            'job_title': df.iloc[idx]['job_title'],
            'similarity': round(1 - dist, 3)
        })
    return pd.DataFrame(results)

# example
recommend_jobs_nn(["python", "machine learning", "deep learning", "nlp"])

,job_title,similarity
0,Co-Op Researcher - Applied Ml/Nlp,0.632
1,Ai/Ml Scientist,0.525
2,"Ai Research Engineer, Large Language Model",0.524
3,Tech Lead - Conversational Ai,0.509
4,Data Scientist,0.504
5,2024 University Graduate - Research Scientist/...,0.501
6,Senior Data Scientist,0.501
7,Data Scientist,0.500
8,Machine Learning Scientist,0.495
9,"Systems Engineer, Ai Infrastructure",0.484


### 4.2 Cosine Similarity

In [15]:
# Cosine Similarity(test)
def calculate_precision_cosine(test_cases, top_n=5):
    correct = 0
    for skills_text, expected_title in test_cases.items():
        user_skills = [s.strip() for s in skills_text.split(",")]
        query_vec = prepare_query(user_skills)
        
        similarities = cosine_similarity(query_vec, tfidf_matrix)[0]
        top_indices = similarities.argsort()[-top_n:][::-1]
        
        top_titles = [df.iloc[idx]["job_title"] for idx in top_indices]
        
        if any(expected_title.lower() in title.lower() for title in top_titles):
            correct += 1
    return correct / len(test_cases) * 100


In [16]:
# final model Cosine Similarity
def recommend_jobs_cosine(user_skills, top_n=10):
    """
    Takes a list of user skills and returns the top matching job titles
    using direct Cosine Similarity against the full TF-IDF matrix.

    Parameters:
        user_skills (list of str): e.g. ["python", "machine learning", "nlp"]
        top_n (int): number of recommendations to return

    Returns:
        pandas.DataFrame with columns: job_title, similarity
    """
    query_vec = prepare_query(user_skills)

    similarities = cosine_similarity(query_vec, tfidf_matrix)[0]
    top_indices = similarities.argsort()[-top_n:][::-1]

    results = []
    for idx in top_indices:
        results.append({
            'job_title': df.iloc[idx]['job_title'],
            'similarity': round(similarities[idx], 3)
        })
    return pd.DataFrame(results)

# example
recommend_jobs_cosine(["python", "machine learning", "deep learning", "nlp"])
 

,job_title,similarity
0,Co-Op Researcher - Applied Ml/Nlp,0.632
1,Ai/Ml Scientist,0.525
2,"Ai Research Engineer, Large Language Model",0.524
3,Tech Lead - Conversational Ai,0.509
4,Data Scientist,0.504
5,2024 University Graduate - Research Scientist/...,0.501
6,Senior Data Scientist,0.501
7,Data Scientist,0.500
8,Machine Learning Scientist,0.495
9,"Systems Engineer, Ai Infrastructure",0.484


### 4.3 K-means

TF-IDF creates a high-dimensional sparse matrix, so TruncatedSVD is applied before K-Means to reduce dimensionality and improve clustering efficiency.

In [17]:
svd = TruncatedSVD(n_components=100, random_state=42)
reduced_matrix = svd.fit_transform(tfidf_matrix)

kmeans = MiniBatchKMeans(n_clusters=200, random_state=42, batch_size=10000)
df["cluster"] = kmeans.fit_predict(reduced_matrix)

print("KMeans fitted successfully")
print(df['cluster'].value_counts().describe())

KMeans fitted successfully
count      200.00000
mean      6470.66000
std       6123.23004
min        852.00000
25%       3378.50000
50%       4949.50000
75%       8174.50000
max      74300.00000
Name: count, dtype: float64


In [18]:
# KMeans + SVD (test)
def calculate_precision_kmeans(test_cases, top_n=5):
    correct = 0
    for skills_text, expected_title in test_cases.items():
        user_skills = [s.strip() for s in skills_text.split(",")]
        query_vec = prepare_query(user_skills)
        query_vec_reduced = svd.transform(query_vec)

        user_cluster = kmeans.predict(query_vec_reduced)[0]
        cluster_indices = df[df["cluster"] == user_cluster].index

        similarities = cosine_similarity(query_vec, tfidf_matrix[cluster_indices]).flatten()
        top_idx = similarities.argsort()[-top_n:][::-1]

        top_titles = [df.loc[cluster_indices[i], "job_title"] for i in top_idx]

        if any(expected_title.lower() in title.lower() for title in top_titles):
            correct += 1
    return correct / len(test_cases) * 100


In [19]:
# final model KMeans (MiniBatch + TruncatedSVD)
svd = TruncatedSVD(n_components=100, random_state=42)
reduced_matrix = svd.fit_transform(tfidf_matrix)

kmeans = MiniBatchKMeans(n_clusters=200, random_state=42, batch_size=10000)
df["cluster"] = kmeans.fit_predict(reduced_matrix)

def recommend_jobs_kmeans(user_skills, top_n=10):
    """
    Takes a list of user skills and returns the top matching job titles
    using MiniBatchKMeans (on SVD-reduced TF-IDF) + Cosine Similarity
    within the assigned cluster.
    """
    query_vec = prepare_query(user_skills)
    query_vec_reduced = svd.transform(query_vec)

    user_cluster = kmeans.predict(query_vec_reduced)[0]
    cluster_indices = df[df["cluster"] == user_cluster].index

    similarities = cosine_similarity(query_vec, tfidf_matrix[cluster_indices]).flatten()
    top_idx = similarities.argsort()[-top_n:][::-1]

    results = []
    for i in top_idx:
        row_idx = cluster_indices[i]
        results.append({
            'job_title': df.loc[row_idx, 'job_title'],
            'similarity': round(similarities[i], 3)
        })
    return pd.DataFrame(results)

recommend_jobs_kmeans(["python", "machine learning", "deep learning", "nlp"])

,job_title,similarity
0,Co-Op Researcher - Applied Ml/Nlp,0.632
1,Ai/Ml Scientist,0.525
2,2024 University Graduate - Research Scientist/...,0.501
3,Senior Data Scientist,0.501
4,Data Scientist,0.500
5,"Systems Engineer, Ai Infrastructure",0.484
6,Machine Learning Engineer,0.471
7,Applied Researcher I,0.449
8,Nlp Team Lead Engineer,0.446
9,Machine Learning Scientist,0.436


## Model Selection

In [20]:
# Precision@5
nn_score = calculate_precision_nn(nn, test_cases, top_n=5)
cosine_score = calculate_precision_cosine(test_cases, top_n=5)
kmeans_score = calculate_precision_kmeans(test_cases, top_n=5)

print("=" * 45)
print("Precision@5")
print("=" * 45)
print(f"NearestNeighbors:   {nn_score:.1f}%")
print(f"Cosine Similarity:  {cosine_score:.1f}%")
print(f"KMeans (SVD):       {kmeans_score:.1f}%")

# MRR 
def calculate_mrr(recommend_function, test_cases, top_n=10):
    reciprocal_ranks = []
    for skills_text, expected_title in test_cases.items():
        user_skills = [s.strip() for s in skills_text.split(",")]
        recs = recommend_function(user_skills, top_n=top_n)
        titles = recs['job_title'].tolist()
        rank = None
        for i, title in enumerate(titles):
            if expected_title.lower() in title.lower():
                rank = i + 1
                break
        reciprocal_ranks.append(1 / rank if rank else 0)
    return sum(reciprocal_ranks) / len(reciprocal_ranks)

mrr_nn = calculate_mrr(recommend_jobs_nn, test_cases)
mrr_cosine = calculate_mrr(recommend_jobs_cosine, test_cases)
mrr_kmeans = calculate_mrr(recommend_jobs_kmeans, test_cases)

print("\n" + "=" * 45)
print("MRR (Mean Reciprocal Rank)")
print("=" * 45)
print(f"NearestNeighbors:   {mrr_nn:.3f}")
print(f"Cosine Similarity:  {mrr_cosine:.3f}")
print(f"KMeans (SVD):       {mrr_kmeans:.3f}")

# Best model
best_model = max([
    ("NearestNeighbors", nn_score),
    ("Cosine Similarity", cosine_score),
    ("KMeans (SVD)", kmeans_score)
], key=lambda x: x[1])

print("\n" + "=" * 45)
print(f"Best model: {best_model[0]} with {best_model[1]:.1f}% Precision@5")
print("=" * 45)

Precision@5
NearestNeighbors:   45.0%
Cosine Similarity:  45.0%
KMeans (SVD):       27.5%

MRR (Mean Reciprocal Rank)
NearestNeighbors:   0.287
Cosine Similarity:  0.287
KMeans (SVD):       0.213

Best model: NearestNeighbors with 45.0% Precision@5


In [21]:
# NN and Cosine

for skills_text, expected_title in test_cases.items():
    user_skills = [s.strip() for s in skills_text.split(",")]
    
    nn_results = recommend_jobs_nn(user_skills, top_n=5)['job_title'].tolist()
    cosine_results = recommend_jobs_cosine(user_skills, top_n=5)['job_title'].tolist()
    
    match = nn_results == cosine_results
    print(f"\nSkills: {skills_text}")
    print(f"  NearestNeighbors: {nn_results}")
    print(f"  Cosine Similarity: {cosine_results}")
    print(f"  Identical? {match}")


Skills: python, sql, excel
  NearestNeighbors: ['Python Developer', 'Business Operations Analyst', 'Denodo Architect - Green Bay, Wi - 12+ Months', 'Audit Methodology Lead', 'Accountant']
  Cosine Similarity: ['Python Developer', 'Business Operations Analyst', 'Denodo Architect - Green Bay, Wi - 12+ Months', 'Audit Methodology Lead', 'Accountant']
  Identical? True

Skills: machine learning, deep learning, nlp
  NearestNeighbors: ['Co-Op Researcher - Applied Ml/Nlp', 'Ai Research Engineer, Large Language Model', 'Machine Learning Operations Engineer', 'Tech Lead - Conversational Ai', 'Data Scientist']
  Cosine Similarity: ['Co-Op Researcher - Applied Ml/Nlp', 'Ai Research Engineer, Large Language Model', 'Machine Learning Operations Engineer', 'Tech Lead - Conversational Ai', 'Data Scientist']
  Identical? True

Skills: react, javascript, css
  NearestNeighbors: ['100% Onsite:  React Developer', 'React Developer', 'Technical Developer – D365 Ecommerce', 'Software Engineer Iv', 'Full S

In [22]:
identical_count = 0
total = len(test_cases)

for skills_text, expected_title in test_cases.items():
    user_skills = [s.strip() for s in skills_text.split(",")]
    nn_results = recommend_jobs_nn(user_skills, top_n=5)['job_title'].tolist()
    cosine_results = recommend_jobs_cosine(user_skills, top_n=5)['job_title'].tolist()
    if nn_results == cosine_results:
        identical_count += 1

print(f"Identical results: {identical_count} out of {total} test cases")

Identical results: 31 out of 40 test cases


##  Model Selection: NearestNeighbors

After building three recommendation models (NearestNeighbors, direct 
Cosine Similarity, and KMeans with TruncatedSVD), we compared their 
performance using two metrics on a set of 40 test cases: 10 manually 
designed cases plus 30 additional cases sampled from the dataset 
itself (a random job posting's real skills used as the query, its 
title used as the expected match), to reduce evaluation bias.

| Model | Precision@5 | MRR |
|---|---|---|
| **NearestNeighbors** | **45.0%** | **0.287** |
| Cosine Similarity | 45.0% | 0.287 |
| KMeans (SVD) | 27.5% | 0.213 |

### Why we chose NearestNeighbors

- **Highest accuracy**: It achieved the best performance among the 
  three models on both metrics.
- **Equivalence with Cosine Similarity**: We verified that 
  NearestNeighbors and Cosine Similarity produce identical top-5 
  results in 31 out of 40 test cases (77.5%), confirming that they 
  are largely equivalent (both rely on Cosine Distance). The 
  remaining differences stem from ties in cosine distance among 
  jobs with very similar or identical skill sets, where minor 
  floating-point precision or tie-breaking order causes a different 
  ranking among equally-similar candidates.
- **Computational efficiency**: Since accuracy was identical, we 
  favored NearestNeighbors because it builds a search index during 
  training, making it faster to respond to each new query — which 
  matters most for an interactive user-facing interface.
- **Comparison with KMeans**: Although KMeans performed reasonably 
  well after fixing the cluster imbalance issue (using TruncatedSVD), 
  its restricted search within a single cluster led to lower accuracy. 
  This gap widened further once evaluated on the larger 40-case set 
  (27.5% vs. 50.0% Precision@5), suggesting the cluster-restricted 
  search generalizes worse than searching the full index.

**Conclusion**: **NearestNeighbors** was adopted as the final model 
used in the recommendation interface.

## Skill Gap

The Skill Gap module identifies frequently required skills among similar jobs to help users understand which skills may improve their profile.

In [23]:
def recommend_with_skill_gap(user_skills, top_n=10, gap_top_n=5, n_similar=30):
    """
    Combines job recommendations with a skill gap analysis for each
    recommended job, using similarity-based skill aggregation.

    For each recommended job, finds its n_similar most similar postings
    (via the pre-fitted NearestNeighbors index) and identifies the most
    common skills among them that the user doesn't already have.

    All similarity lookups for the recommended jobs are batched into a
    single kneighbors call for performance.
    """
    query_vec = prepare_query(user_skills)
    distances, indices = nn.kneighbors(query_vec, n_neighbors=top_n)
    job_indices = indices[0]

    batch_vectors = tfidf_matrix[job_indices]
    batch_distances, batch_neighbor_indices = nn.kneighbors(batch_vectors, n_neighbors=n_similar + 1)

    user_skills_clean = set([skill.strip().lower() for skill in user_skills])

    results = []
    for row, job_idx, dist in zip(range(len(job_indices)), job_indices, distances[0]):
        neighbor_idx = batch_neighbor_indices[row]
        neighbor_idx = neighbor_idx[neighbor_idx != job_idx][:n_similar]

        similar_skills = df.iloc[neighbor_idx]['job_skills'].explode()
        top_skills = similar_skills.value_counts().head(gap_top_n).index.tolist()
        top_skills_clean = set([s.lower() for s in top_skills])

        missing = top_skills_clean - user_skills_clean
        gap = sorted(missing) if missing else "No major gaps found"

        results.append({
            'job_title': df.iloc[job_idx]['job_title'],
            'similarity': round(1 - dist, 3),
            'missing_skills': gap
        })
    return pd.DataFrame(results)


# Example
result = recommend_with_skill_gap(["python", "sql", "r"], top_n=10, gap_top_n=5)

for _, row in result.iterrows():
    print(f"\n{row['job_title']} - similarity: {row['similarity']}")
    print(f"  Missing skills: {row['missing_skills']}")


Python Developer - similarity: 0.732
  Missing skills: ['communication', 'data analysis', 'tableau']

Fraud Risk Analyst - similarity: 0.71
  Missing skills: ['communication', 'data analysis']

Marketing Specialist 3 Company Hidden Technology Services Redmond, Wa 1 Opening - similarity: 0.609
  Missing skills: ['data analysis', 'tableau']

Risk Management Consultant - similarity: 0.583
  Missing skills: ['analytical and quantitative', 'communication']

Senior Data Analyst - similarity: 0.556
  Missing skills: ['data science', 'machine learning']

Senior Manager, Marketing Strategy & Analytics - similarity: 0.549
  Missing skills: ['data visualization', 'tableau']

Customer Facing Research Analyst - similarity: 0.539
  Missing skills: ['data analysis', 'statistics']

Planner - 1St Shift - similarity: 0.538
  Missing skills: ['excel', 'vba']

Manager, Supply Analytics - similarity: 0.529
  Missing skills: ['data visualization', 'tableau']

Senior Data Analyst - similarity: 0.525
  Missi

## Conclusion & Limitations

This notebook covered the full pipeline: data cleaning, TF-IDF 
representation, three candidate models compared via Precision@5 and 
MRR, and a similarity-based Skill Gap feature. NearestNeighbors was 
selected as the final model for its accuracy and query-time 
efficiency.

While the system is fully functional, we want to be transparent 
about a few methodological choices and known limitations, since 
being upfront about them reflects the scientific rigor we aimed for 
throughout this project.

---

**Evaluation set size.** Precision@5 and MRR are computed on 40 test 
cases — 10 manually designed plus 30 sampled from real postings in 
the dataset, added specifically to reduce evaluation bias. This is a 
meaningful improvement over a single hand-picked set, but still a 
modest sample relative to 1.29M job postings. The resulting 
percentages are best read as directional estimates of relative model 
performance, not tight statistical guarantees.

**Hyperparameter selection.** Key parameters — `min_df=15` for 
TF-IDF, `n_similar=30` for Skill Gap, `n_components=100` for SVD, and 
`n_clusters=200` for KMeans — were chosen by comparing a small set of 
candidate values against the same test cases used for final 
evaluation, rather than through formal cross-validation on a held-out 
set. This is a practical choice for an MVP timeline, though it 
carries some risk of the settings being mildly tuned to the 
evaluation set itself.

**Match evaluation method.** A recommendation is scored as "correct" 
using substring matching between predicted and expected job titles. 
Given how specific and long-tail the job titles in this dataset are, 
this proved to be the most practical proxy available — though it can 
occasionally over- or under-count matches for generic or 
differently-worded titles.

**Data quality.** The raw skills data contained noise inherited from 
the original job postings (stray punctuation and isolated digits). 
We removed this noise while explicitly preserving two single-character 
skills that are easy to lose in this kind of cleanup but carry real 
signal: `r` and `c`, the R and C programming languages.

---

*These are documented as areas for future refinement rather than 
flaws — each reflects a deliberate trade-off appropriate for the 
current stage of the project.*